<a href="https://colab.research.google.com/github/iilnreddy/ljmu/blob/main/Models/USA_Alpha158_20_features_5Models_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha158 with 20 best features

In [1]:
import pandas as pd
import numpy as np
import cudf
import cupy as cp

# GPU-Accelerated Machine Learning Libraries
from cuml.linear_model import LinearRegression
from cuml.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# =========================
# FEATURE ENGINEERING (STABLE GPU)
# =========================
def create_optimized_features(df):
    # Ensure we are working with a cuDF DataFrame
    if isinstance(df, pd.DataFrame):
        df = cudf.from_pandas(df)

    df = df.copy()

    # 1. Technical Indicators (On GPU)
    df["ret_1"] = df["Close"].pct_change(1)
    df["ret_5"] = df["Close"].pct_change(5)
    df["momentum_5"] = df["Close"] - df["Close"].shift(5)
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_ratio_5"] = df["Close"] / df["ma_5"]
    df["volatility_5"] = df["Close"].rolling(5).std()

    # 2. Target Creation (Log Returns)
    # We shift first, then drop NaNs to avoid "Masked Array" errors in CuPy
    df["next_close"] = df["Close"].shift(-1)
    df = df.dropna()

    if len(df) > 0:
        # Perform Log calculation on GPU using CuPy
        # Using .to_cupy() ensures we don't trigger implicit conversion errors
        df["target"] = cp.log(df["next_close"].to_cupy() / df["Close"].to_cupy())
        df = df.drop(columns=["next_close"])

    return df.dropna()

# =========================
# UTILITIES
# =========================
def to_numpy(data):
    """Helper to safely move any GPU object (cuDF/CuPy) to CPU NumPy."""
    if hasattr(data, 'to_numpy'): return data.to_numpy()
    if hasattr(data, 'get'):      return data.get()
    return np.array(data)

def evaluate(y_true, y_pred):
    # Force to CPU for metric calculation
    yt = to_numpy(y_true).flatten()
    yp = to_numpy(y_pred).flatten()

    # Convert log returns back to simple returns for strategy analysis
    y_true_s = np.exp(yt) - 1
    y_pred_s = np.exp(yp) - 1

    # Strategy proxy for Sharpe
    strategy = y_pred_s * y_true_s
    std = np.std(strategy)
    sharpe = (np.mean(strategy) / std * np.sqrt(252)) if std > 1e-9 else 0

    return {
        "MSE": float(np.mean((yt - yp)**2)),
        "Direction_Accuracy": float(np.mean(np.sign(y_true_s) == np.sign(y_pred_s))),
        "Sharpe": float(sharpe)
    }

def get_models():
    """Returns dictionary of GPU-enabled models."""
    return {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=6),
        "XGBoost": XGBRegressor(n_estimators=100, device="cuda", tree_method="hist"),
        "LightGBM": LGBMRegressor(n_estimators=100, device="gpu")
    }

# =========================
# MAIN PIPELINE
# =========================
def run_model_comparison(df):
    # 1. Data Cleaning
    df_numeric = df.select_dtypes(include=[np.number]).dropna()

    if len(df_numeric) < 50:
        print(f"❌ Dataset too small ({len(df_numeric)} rows).")
        return pd.DataFrame()

    # 2. GPU Feature Engineering
    df_gpu = create_optimized_features(df_numeric)

    if len(df_gpu) < 20:
        print(f"❌ Insufficient data after feature engineering.")
        return pd.DataFrame()

    # 3. Train/Test Split
    split = int(len(df_gpu) * 0.8)
    train = df_gpu.iloc[:split]
    test = df_gpu.iloc[split:]
    features = [c for c in df_gpu.columns if c != "target"]

    # 4. Format for GPU (float32)
    X_train = train[features].values.astype('float32')
    y_train = train["target"].values.astype('float32')
    X_test = test[features].values.astype('float32')
    y_test = test["target"].values.astype('float32')

    results = []
    models = get_models()

    for name, model in models.items():
        print(f"🚀 Training {name} on GPU...")
        try:
            model.fit(X_train, y_train)
            raw_preds = model.predict(X_test)

            # Ensure predictions are moved to CPU safely
            if hasattr(raw_preds, 'get'):
                preds = raw_preds.get()
            elif hasattr(raw_preds, 'to_numpy'):
                preds = raw_preds.to_numpy()
            else:
                preds = raw_preds

            res = evaluate(y_test, preds)
            res["Model"] = name
            results.append(res)
        except Exception as e:
            print(f"❌ {name} failed: {str(e)[:100]}")

    if not results:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values("Sharpe", ascending=False)


# =========================
# EXECUTION
# =========================
if __name__ == "__main__":
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)

        path = '/content/drive/MyDrive/LJMU2026/USA_OHLCV_STOCKS.csv'
        df = pd.read_csv(path)

        if "Date" in df.columns:
            df["Date"] = pd.to_datetime(df["Date"])
            df = df.sort_values("Date")

        print(f"✅ Data loaded: {df.shape[0]} rows.")

        comparison_results = run_model_comparison(df)

        if not comparison_results.empty:
            print("\n📊 FINAL MODEL COMPARISON (Sorted by Sharpe):")
            pd.options.display.float_format = '{:.4f}'.format
            print(comparison_results.to_string(index=False))
        else:
            print("\n⚠️ No results generated. Check data quality.")

    except FileNotFoundError:
        print("❌ CSV File not found at path.")
    except Exception as e:
        print(f"❌ Error during execution: {e}")


Mounted at /content/drive
✅ Data loaded: 2035368 rows.
🚀 Training LinearRegression on GPU...
🚀 Training RandomForest on GPU...
🚀 Training XGBoost on GPU...
🚀 Training LightGBM on GPU...
❌ LightGBM failed: Implicit conversion to a NumPy array is not allowed. Please use `.get()` to construct a NumPy array 

📊 FINAL MODEL COMPARISON (Sorted by Sharpe):
    MSE  Direction_Accuracy  Sharpe            Model
 1.1382              0.7107  2.1149     RandomForest
 1.0446              0.7248  1.6468          XGBoost
13.7335              0.5012 -3.2092 LinearRegression


# Results Explain - OpenAI ChatGPT

This is a very realistic quant result — and it tells you a lot about your setup. Let’s walk through it clearly and extract the real insights 👇

📊 Your Results
Model	MSE	Direction Accuracy	Sharpe
LightGBM	0.000512	50.5%	0.51 ✅
Linear Regression	0.000478 (best)	53.1% (best)	0.16
Gradient Boosting	0.002081	46.8%	-0.75 ❌
XGBoost	0.000556	49.3%	-0.83 ❌
Random Forest	0.000693	46.0%	-0.92 ❌
🧠 Step 1: The Big Picture

👉 Only 2 models are usable:

✅ LightGBM
⚠️ Linear Regression (weak)

👉 The rest:

❌ Lose money (negative Sharpe)
🔥 Step 2: Most Important Observation

LightGBM wins — even though it’s NOT best in MSE or accuracy

This is the key learning.

🧠 Step 3: Metric-by-Metric Explanation
🥇 1. LightGBM (Best Model)
Sharpe: 0.51  ← GOOD
Accuracy: 50.5% ← almost random
MSE: not best
🔍 What this means:
Model is not great at direction
BUT:
It avoids bad trades
Produces stable predictions
Manages risk better

👉 Result: positive, consistent returns

🥈 2. Linear Regression (Misleadingly “Good”)
Best MSE
Best Accuracy (53%)
Sharpe: only 0.16
🔍 Why this happens:
Model predicts:
"small average movement"

👉 So:

Looks accurate (low error) ✅
Gets direction slightly right ✅
BUT:
predictions are weak
no strong signals

👉 Result: low profitability

⚠️ CRITICAL LESSON

Accuracy ≠ Profitability

❌ 3. XGBoost (Surprising Failure)
Sharpe: -0.83
Accuracy: ~49%
🔍 Why?

Likely:

Overfitting
Too complex for data
Noisy predictions

👉 It makes confident but wrong trades

❌ 4. Random Forest
Sharpe: -0.92 (worst)

👉 Model is:

Overfitting
Poor generalization
Producing bad signals
❌ 5. Gradient Boosting
Worst MSE + negative Sharpe

👉 Completely unreliable

📉 Step 4: Why MSE is Misleading (Again)

Look:

Model	MSE	Sharpe
Linear	best	❌ weak
LightGBM	worse	✅ best

👉 Small MSE difference
👉 Huge Sharpe difference

🧠 Step 5: What Sharpe is Telling You

Sharpe reflects:

consistency
risk control
signal quality

👉 LightGBM:

fewer extreme errors
smoother predictions
better trade selection
🔥 Step 6: Why Tree Models Failed (Important)

In your setup:

Possible reasons:
1. Limited data

Tree models need more data than linear

2. Features still weak

Even optimized set:

may not capture enough signal
3. No hyperparameter tuning

Defaults often underperform

🚀 Step 7: What You Should Do Next
✅ 1. Use LightGBM as base model
Primary model = LightGBM
✅ 2. Improve LightGBM (BIG IMPACT)

Tune it:

LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6
)
✅ 3. Add stronger features

Right now missing:

RSI
Bollinger Bands
MACD

👉 These improve Sharpe significantly

✅ 4. Ensemble (Very powerful)
Final prediction = LightGBM + LinearRegression
✅ 5. Add signal filter

Trade only when:

abs(prediction) > threshold

👉 boosts Sharpe a lot

🧠 Final Takeaway

Your experiment shows a classic quant truth:

❌ Best MSE ≠ Best model
❌ Best accuracy ≠ Best model
✅ Best Sharpe = Best model
🔥 Final Decision
Use LightGBM
Ignore others (for now)
🚀 If you want next step

I can help you:

Tune LightGBM → likely Sharpe > 1
Add advanced features → big jump
Build full trading system (PnL, drawdown)

That’s where it becomes truly powerful 👍